# Imports

In [21]:


import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

from tifffile import imread, imwrite as imsave
from csbdeep.utils import Path, normalize
from csbdeep.utils.tf import keras_import
from csbdeep.data import PercentileNormalizer
keras = keras_import()
import tensorflow as tf

from stardist import export_imagej_rois, random_label_cmap
from stardist.models import StarDist2D
import os
from joblib import Parallel, delayed

np.random.seed(0)
cmap = random_label_cmap()

In [17]:
input_dir = "/mnt/towbin.data/shared/spsalmon/towbinlab_segmentation_database/stardist/emr1_panoptic_dataset_60x_auto_seg/raw"
image_paths = [os.path.join(input_dir, f) for f in os.listdir(input_dir)]
image = imread(image_paths[90])[5:15, 1]

In [25]:
model = StarDist2D(None, name='emr1_60x', basedir='/mnt/towbin.data/shared/spsalmon/towbinlab_segmentation_database/stardist/')
print(model)
mask = np.zeros_like(image, dtype=np.uint16)
normalizer = PercentileNormalizer(pmin=1, pmax=99.8, do_after=False)

for i, plane in enumerate(image):
    labels, _ = model.predict_instances(plane, prob_thresh=0.6, normalizer=normalizer)
    mask[i] = labels

Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.458353, nms_thresh=0.3.
StarDist2D(emr1_60x): YXC → YXC
├─ Directory: /mnt/towbin.data/shared/spsalmon/towbinlab_segmentation_database/stardist/emr1_60x
└─ Config2D(n_dim=2, axes='YXC', n_channel_in=1, n_channel_out=33, train_checkpoint='weights_best.h5', train_checkpoint_last='weights_last.h5', train_checkpoint_epoch='weights_now.h5', n_rays=32, grid=(1, 1), backbone='unet', n_classes=None, unet_n_depth=3, unet_kernel_size=[3, 3], unet_n_filter_base=32, unet_n_conv_per_depth=2, unet_pool=[2, 2], unet_activation='relu', unet_last_activation='relu', unet_batch_norm=False, unet_dropout=0.0, unet_prefix='', net_conv_after_unet=128, net_input_shape=[None, None, 1], net_mask_shape=[None, None, 1], train_shape_completion=False, train_completion_crop=32, train_patch_size=[512, 512], train_background_reg=0.0001, train_foreground_only=0.9, train_sample_cache=True, train

In [24]:
def process_plane(plane, model, normalizer):
    labels, _ = model.predict_instances(plane, prob_thresh=0.6, normalizer=normalizer)
    return labels

mask = np.array(Parallel(n_jobs=8)(delayed(process_plane)(plane, model, normalizer) for plane in image))